**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**02 | HARTH Exploratory Classical Machine Learning and Subject-Independent Evaluation**

![Consolidated HARTH Stage 02 research workflow](../../images/02_consolidated_workflow.svg)

**Objective:** Integrate four related but distinct studies: eight-person source inspection and participant splitting, an early three-model Validation comparison, six-person Development cross-validation and expanded ten-person Development cross-validation.

**Dataset:** The official pinned NTNU HARTH v2.0 release, using two tri-axial accelerometers at 50 Hz. Label-pure two-second windows use a 50-sample stride and five statistical measures per channel. The recorded Stage 01 feature CSV provides S023 without reprocessing; the other eleven distinct participants are processed only once in this notebook.

**Evaluation boundary:** The original eight-person pilot uses four Train, two Validation and two exploratory reserved participants. Six Development subjects are evaluated in three GroupKFold splits; ten Development subjects are evaluated in five. The supervised task uses documented activity codes 1–8 only. Out-of-task and undocumented source codes are audited after windowing, then excluded from modelling. S023 and S032 are never scored here and were already inspected descriptively in earlier work.

**Reading order:** Stage 01 feature reuse → remaining participant source processing → eight-person source-label audit and pilot partition → three classical Validation models → six-person grouped CV → ten-person coverage and grouped CV → saved current research outputs.

**Output contract:** Stage 02 saves current CSV tables and PNG plots under the 02_ prefix in the study output directory. A ten-person Development feature CSV provides reusable input for the next stage. No historical version comparisons, approval manifests or administrative quality gates are required.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# --------------------------- 1. Study Environment ----------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ------------------------ 2. Scientific Data Analysis ------------------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay
from core.plot_style import plot_style, save_figure
plot_style(font_family="DejaVu Sans", title_size=12.5,
           title_weight="normal", label_size=11.5,
           label_weight="normal", tick_size=11, legend_size=10,
           figure_dpi=100, figure_size=(9, 4))

# ---------------- 4. Shared Source, Split and Model Utilities ----------------
from core.signal_preprocessing import create_signal_windows
from core.feature_engineering import extract_signal_features
from core.data_splitting import split_group_indices
from core.ml_baselines import (
    train_validation_classifiers, classification_class_report,
    classification_confusion,
)
from core.group_cross_validation import evaluate_group_cross_validation
from core.output_download import download_outputs_from_colab
from harth_data import (
    ACTIVITY_NAMES, SENSOR_COLUMNS, ensure_harth_sample,
    load_harth_subject,
)

# ---------------- 5. Research Source and Output Configuration ----------------
output_dir = root / "wearable_activity_recognition/output"
data_dir = root / "data/raw/harth_v2"
output_dir.mkdir(parents=True, exist_ok=True)
SUBJECTS = ["S023", "S024", "S025", "S027", "S028",
            "S032", "S034", "S035"]
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SIX = ["S024", "S025", "S027", "S028", "S034", "S035"]
DEVELOPMENT_SUBJECTS = DEVELOPMENT_SIX + ["S029", "S030", "S031", "S033"]
ALL_SUBJECTS = list(dict.fromkeys(SUBJECTS + DEVELOPMENT_SUBJECTS))
TASK_CLASSES = tuple(range(1, 9))
FEATURE_METHODS = ["mean", "std", "rms", "peak_to_peak", "waveform_length"]
SAMPLING_RATE, WINDOW_SIZE, STEP_SIZE = 50, 100, 50
print("Unique source participants:", len(ALL_SUBJECTS))
print("Research output directory:", output_dir)

**02. Read the Aligned Stage 01 Feature Output for S023**

Stage 01 already extracted S023 windows. Reuse its CSV rather than downloading S023 and recomputing the same features. Preserve the original source-row indices, activity labels and feature column order.

In [ ]:
# ================= 02. Reuse the Current S023 Feature Matrix =================

# ------------- 1. Read Stage 01 Feature Rows and Source Summary --------------
stage01 = pd.read_csv(output_dir / "01_statistical_features.csv")
stage01_summary = pd.read_csv(output_dir / "01_source_inspection_summary.csv")
feature_cols = [f"{c}__{m}" for c in SENSOR_COLUMNS for m in FEATURE_METHODS]
window_names = {
    "window_subject_id": "subject_id",
    "window_start_index": "start_index",
    "window_end_index_exclusive": "end_index_exclusive",
    "window_label": "label",
    "window_label_purity": "label_purity",
}
needed = set(feature_cols + list(window_names) + ["source_label"])
if needed - set(stage01.columns):
    raise ValueError("Stage 01 feature export is missing expected columns.")
s023_features = stage01[feature_cols].copy()
s023_meta = stage01[list(window_names)].rename(columns=window_names).copy()
if not (s023_meta["subject_id"] == "S023").all():
    raise ValueError("Stage 01 data contain an unexpected subject.")
if not np.array_equal(s023_meta["label"], stage01["source_label"]):
    raise ValueError("Stage 01 source labels and feature rows are misaligned.")
feature_by_subject = {"S023": s023_features}
metadata_by_subject = {"S023": s023_meta}
subject_rows = [{
    "subject_id": "S023", "source_file": "S023.csv (Stage 01)",
    "raw_samples": int(stage01_summary.loc[0, "recorded_samples"]),
    "retained_windows": len(s023_meta),
    "observed_activity_codes": s023_meta["label"].nunique(),
    "unmapped_raw_rows": 0,
}]
print("Reused Stage 01 feature matrix:", s023_features.shape)
display(s023_meta.head())

**03. Prepare Eleven Additional Participants Only Once**

Retrieve missing official source recordings and apply timestamp-aware, label-pure two-second windowing and the five shared feature methods. Preserve undocumented source codes for the audit. The six Development participants common to the original pilot and expanded study are not processed twice.

In [ ]:
# ================ 03. Generate Distinct Participant Features =================

# -------------- 1. Per-Subject Windowing and Feature Extraction --------------
for subject in ALL_SUBJECTS:
    if subject == "S023":
        continue  # Already produced by Stage 01.
    file_path = ensure_harth_sample(data_dir, subject=subject)
    data, channels = load_harth_subject(
        file_path, allow_unmapped_labels=True
    )
    windows, labels, info = create_signal_windows(
        signals=data[channels].to_numpy(),
        labels=data["label"].to_numpy(),
        window_size=WINDOW_SIZE, step_size=STEP_SIZE,
        timestamps=data["timestamp"], sampling_rate=SAMPLING_RATE,
        min_label_purity=1.0, subject_id=subject,
    )
    if not len(windows):
        raise ValueError(f"No valid source windows for {subject}.")
    feature_by_subject[subject] = extract_signal_features(
        windows, channel_names=channels, methods=FEATURE_METHODS
    )
    metadata_by_subject[subject] = info
    subject_rows.append({
        "subject_id": subject, "source_file": file_path.name,
        "raw_samples": len(data), "retained_windows": len(info),
        "observed_activity_codes": info["label"].nunique(),
        "unmapped_raw_rows": int((~data["label"].isin(ACTIVITY_NAMES)).sum()),
    })

# -------------- 2. Aligned Current Features and Source Summary ---------------
features_full = pd.concat(
    [feature_by_subject[s] for s in ALL_SUBJECTS], ignore_index=True
)
metadata_full = pd.concat(
    [metadata_by_subject[s] for s in ALL_SUBJECTS], ignore_index=True
)
if len(features_full) != len(metadata_full):
    raise ValueError("Source metadata and features are misaligned.")
if list(features_full.columns) != feature_cols:
    raise ValueError("Feature columns differ across participants.")
source_summary = pd.DataFrame(subject_rows)
display(source_summary)
source_summary.to_csv(output_dir / "02_source_participant_summary.csv", index=False)
print("Source feature matrix:", features_full.shape)

**04. Audit Original Eight-Person Source-Label Coverage**

Select the original eight participants from memory, not by reconstructing their windows. Report the recorded source activity codes and flag undocumented labels before selecting the documented eight-class prediction task.

In [ ]:
# ============== 04. Source-Label Audit Before Task Restriction ===============

pilot_mask = metadata_full["subject_id"].isin(SUBJECTS).to_numpy()
features_all = features_full.loc[pilot_mask].reset_index(drop=True)
metadata_all = metadata_full.loc[pilot_mask].reset_index(drop=True)
raw_codes = (metadata_all.groupby(["subject_id", "label"]).size()
             .rename("windows").reset_index())
raw_codes["activity"] = raw_codes["label"].map(
    lambda c: ACTIVITY_NAMES.get(int(c), "UNDOCUMENTED source code")
)
raw_codes["in_task"] = raw_codes["label"].isin(TASK_CLASSES)
display(raw_codes)
raw_codes.to_csv(output_dir / "02_pilot_source_label_audit.csv", index=False)
window_metadata = metadata_all
feature_table = features_all
print("Original pilot feature matrix:", features_all.shape)

**05. Split Whole Pilot Participants and Inspect Activity Distributions**

Repeat the original fixed-seed four/two/two pilot split. Preserve both its activity-by-partition count table and grouped-bar/heatmap figure. These descriptive source distributions include labels outside the eventual supervised task.

In [ ]:
# ================ 04. Create Participant-Disjoint Partitions =================

# ----------------- 1. Create Participant-Disjoint Partitions -----------------
partitions = split_group_indices(
    window_metadata["subject_id"].to_numpy(),
    train_fraction=0.50, validation_fraction=0.25,
    test_fraction=0.25, random_state=42,
)
groups = {
    name: set(window_metadata.iloc[index]["subject_id"])
    for name,index in partitions.items()
}

assert not (groups["train"]&groups["validation"] or
            groups["train"]&groups["test"] or groups["validation"]&groups["test"])

np.testing.assert_array_equal(
    np.sort(np.concatenate(list(partitions.values()))),
    np.arange(len(window_metadata))
)

# ------------------ 2. Check Group Separation and Summarise ------------------
partition_summary = pd.DataFrame([
    {"partition":p,"subjects":len(groups[p]),"windows":len(ix),
     "IDs":", ".join(sorted(groups[p]))}
    for p,ix in partitions.items()
])
display(partition_summary)
partition_summary.to_csv(output_dir / "02_pilot_source_exploratory_partition_membership.csv", index=False)
print("Subject leakage: NONE")


# =============== 05. Compare Partition Activity Distributions ================

# ------------------- 1. Activity Distribution by Partition -------------------
partition_order = ["train", "validation", "test"]
counts = pd.DataFrame({
    part: window_metadata.iloc[partitions[part]]["label"].value_counts()
    for part in partition_order
}).fillna(0).astype(int).sort_index()
counts.index = [
    f"{int(code)}: {ACTIVITY_NAMES.get(int(code), 'UNDOCUMENTED source code')}"
    for code in counts.index
]
display(counts)
counts.rename_axis("activity").reset_index().to_csv(
    output_dir / "02_pilot_source_partition_activity_counts.csv", index=False
)
fractions = counts.div(counts.sum(axis=0), axis=1) * 100
fractions.rename_axis("activity").reset_index().to_csv(
    output_dir / "02_pilot_source_partition_activity_percentages.csv", index=False
)
activity_names = fractions.index.tolist()
n_activities = len(activity_names)
positions = np.arange(len(partition_order))
bar_width = 0.82 / n_activities
colours = plt.get_cmap("tab20")(np.linspace(0, 1, n_activities))

fig, (ax_bars, ax_heatmap) = plt.subplots(
    2, 1, figsize=(18, 12),
    gridspec_kw={"height_ratios": [1.15, 1.35]},
)

# ---------------------- 2. Grouped Activity Comparison -----------------------
for i, (activity, colour) in enumerate(zip(activity_names, colours)):
    x_positions = positions + (i - (n_activities - 1) / 2) * bar_width
    ax_bars.bar(
        x_positions,
        fractions.loc[activity, partition_order].to_numpy(),
        width=bar_width * 0.92,
        label=activity,
        color=colour,
    )

ax_bars.set_xticks(positions, ["Train", "Validation", "Test"])
ax_bars.set(
    title="HARTH: activity proportions by data partition (grouped bars)",
    xlabel="Partition",
    ylabel="Fraction of windows (%)",
    ylim=(0, 100),
)
ax_bars.grid(axis="y", alpha=0.25)
ax_bars.set_axisbelow(True)
ax_bars.legend(
    title="Activity",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

# ---------------------- 3. Activity Proportion Heatmap -----------------------
heat_values = fractions[partition_order].to_numpy()
heatmap = ax_heatmap.imshow(
    heat_values,
    aspect="auto",
    cmap="YlGnBu",
    vmin=0,
    vmax=max(1.0, float(heat_values.max())),
)
ax_heatmap.set_xticks(positions, ["Train", "Validation", "Test"])
ax_heatmap.set_yticks(np.arange(n_activities), activity_names)
ax_heatmap.set(title="HARTH: activity proportions (heatmap)", xlabel="Partition")
for row in range(n_activities):
    for column in range(len(partition_order)):
        value = heat_values[row, column]
        text_colour = "white" if value > heat_values.max() * 0.55 else "black"  # Improve contrast for larger values.
        ax_heatmap.text(
            column, row, f"{value:.1f}%",
            ha="center", va="center", fontsize=8, color=text_colour,
        )
colourbar = fig.colorbar(heatmap, ax=ax_heatmap, fraction=0.035, pad=0.025)
colourbar.set_label("Fraction of windows (%)")
fig.subplots_adjust(left=0.25, right=0.78, top=0.95, bottom=0.06, hspace=0.36)
save_figure(fig, output_dir / "02_pilot_source_exploratory_partition_activity.png")
plt.show()

# ---------------- 4. Missing and Undocumented Activity Codes -----------------
print("Classes absent by partition:", {
    part: counts.index[counts[part] == 0].tolist()
    for part in partition_order
})

unmapped_window_codes = sorted(
    set(window_metadata["label"].unique()) - set(ACTIVITY_NAMES)
)
print("Undocumented codes in retained windows:", unmapped_window_codes)
print("Undocumented codes will be excluded before supervised fitting.")

**06. Restrict Pilot Models to Published Activity Codes 1–8**

Retain only documented HARTH eight-class labels after source-safe windowing. Excluded codes are reported rather than redefined or silently included in classifier training.

In [ ]:
# ============== 03. Restrict to the Documented Eight-Class Task ==============

# ------------------ 1. Audit Source Labels and Select Task -------------------
raw_class_counts = (
    metadata_all.groupby(["subject_id", "label"])
    .size().rename("windows").reset_index()
)

raw_class_counts["activity"] = raw_class_counts["label"].map(
    lambda value: ACTIVITY_NAMES.get(int(value), "UNDOCUMENTED")
)

raw_class_counts["in_task"] = raw_class_counts["label"].isin(TASK_CLASSES)

display(raw_class_counts)
raw_class_counts.to_csv(output_dir / "02_pilot_validation_source_label_audit.csv", index=False)
included = metadata_all["label"].isin(TASK_CLASSES).to_numpy()  # Filter only after constructing time-safe windows.

# ------------------ 2. Report Exclusions and Align Features ------------------
excluded = metadata_all.loc[~included, "label"].value_counts().sort_index()
features = features_all.loc[included].reset_index(drop=True)
metadata = metadata_all.loc[included].reset_index(drop=True)

print("Excluded windows by source code:", excluded.to_dict())
print("Eight-class feature matrix:", features.shape)

**07. Create the Pilot Train and Validation Cohorts**

Fit the original participant-disjoint split on the eligible eight-class subset. Four Train participants provide fitting data, two others provide exploratory Validation, and S023/S032 remain reserved without model inference.

In [ ]:
# ================ 04. Create Subject-Disjoint Data Partitions ================

# ------------------- 1. Create Subject-Disjoint Partitions -------------------
partitions = split_group_indices(
    metadata["subject_id"].to_numpy(),
    train_fraction=0.50, validation_fraction=0.25,
    test_fraction=0.25, random_state=42,
)

# ------------------ 2. Verify Groups and Report Membership -------------------
group_sets = {
    part: set(metadata.iloc[idx]["subject_id"])
    for part, idx in partitions.items()
}

assert not (group_sets["train"] & group_sets["validation"])
assert not (group_sets["train"] & group_sets["test"])
assert not (group_sets["validation"] & group_sets["test"])
partition_summary = pd.DataFrame([
    {"partition":part,"subjects":", ".join(sorted(group_sets[part])),
     "windows":len(idx)}
    for part,idx in partitions.items()
])
display(partition_summary)
partition_summary.to_csv(output_dir / "02_pilot_validation_exploratory_partition_membership.csv", index=False)

# ------------------ 3. Train and Validation Class Coverage -------------------
train_ids, validation_ids = partitions["train"], partitions["validation"]
X_train = features.iloc[train_ids]
y_train = metadata.iloc[train_ids]["label"].to_numpy()
X_validation = features.iloc[validation_ids]
y_validation = metadata.iloc[validation_ids]["label"].to_numpy()
coverage = pd.DataFrame({
    "train": pd.Series(y_train).value_counts(),
    "validation": pd.Series(y_validation).value_counts(),
}).reindex(TASK_CLASSES, fill_value=0).fillna(0).astype(int)
coverage.index = [ACTIVITY_NAMES[c] for c in TASK_CLASSES]

display(coverage)
coverage.rename_axis("activity").reset_index().to_csv(
    output_dir / "02_pilot_validation_train_validation_class_coverage.csv", index=False
)
assert (coverage["train"] > 0).all(), "Not every defined task class is in Train."

print("Held-out test partition: reserved, not inspected for modelling")

**08. Fit Dummy, Logistic Regression and Random Forest**

Reuse the original shared classical baseline interface with its Train-only Logistic Regression StandardScaler and class-balanced Random Forest. Report fixed eight-class Macro F1 on the pilot Validation participants only.

In [ ]:
# ================ 05. Train and Compare Validation Baselines =================

# ------------------- 1. Fit and Compare Validation Models --------------------
models, comparison, predictions = train_validation_classifiers(
    X_train, y_train, X_validation, y_validation,
    classes=TASK_CLASSES, random_state=42,
)
display(comparison.style.format({
    name:"{:.3f}" for name in comparison.columns if name != "model"
}))
comparison.to_csv(output_dir / "02_pilot_validation_validation_model_metrics.csv", index=False)
best_name = comparison.loc[0, "model"]

print("Best validation Macro F1:", best_name)
print("Final Test evaluated: NO")

**09. Visualise Pilot Validation and Inspect Per-Class Errors**

Preserve the pilot metric comparison chart, per-class precision/recall/F1 table, original participant-indexed predictions and confusion matrix for the Validation-selected model.

In [ ]:
# ================== 06. Compare Validation Metrics Visually ==================
plot_table = comparison.set_index("model")[["macro_f1", "balanced_accuracy"]]
ax = plot_table.plot(kind="bar", figsize=(9, 4.5), rot=0)
ax.set(title="HARTH: validation-only classifier comparison",
       xlabel="Model", ylabel="Score (0 to 1)", ylim=(0, 1))
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
save_figure(ax.figure, output_dir / "02_pilot_validation_validation_model_comparison.png")
plt.show()


# ================= 07. Inspect Validation Class-Level Errors =================

# ----------------- 1. Selected Model and Class-Level Metrics -----------------
best_predictions = predictions[best_name]
class_names = [ACTIVITY_NAMES[code] for code in TASK_CLASSES]
class_report = classification_class_report(
    y_validation, best_predictions,
    classes=TASK_CLASSES, class_names=class_names,
)

display(class_report)
class_report.to_csv(output_dir / "02_pilot_validation_selected_validation_per_class.csv", index=False)
pd.DataFrame({
    "subject_id": metadata.iloc[validation_ids]["subject_id"].to_numpy(),
    "true_source_code": y_validation,
    "predicted_source_code": best_predictions,
}).to_csv(output_dir / "02_pilot_validation_selected_validation_predictions.csv", index=False)

# ---------------------- 2. Validation Confusion Matrix -----------------------
matrix = classification_confusion(
    y_validation, best_predictions, classes=TASK_CLASSES
)
pd.DataFrame(matrix, index=class_names, columns=class_names).rename_axis(
    "true_activity"
).to_csv(output_dir / "02_pilot_validation_selected_validation_confusion_counts.csv")

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(
    confusion_matrix=matrix, display_labels=class_names
).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title(f"HARTH Validation confusion matrix: {best_name}")
plt.tight_layout()
save_figure(fig, output_dir / "02_pilot_validation_selected_validation_confusion.png")
plt.show()

**10. Evaluate Six Development Participants with Three Grouped Folds**

Merge the four pilot Train and two pilot Validation participant sets into six-person Development, excluding the reserved subjects. Fit the three classifiers independently in each participant-disjoint GroupKFold split.

In [ ]:
# =================== 10. Six-Person Development GroupKFold ===================

development_indices = np.sort(np.concatenate([
    partitions["train"], partitions["validation"]
]))
development_features = features.iloc[development_indices].to_numpy()
development_labels = metadata.iloc[development_indices]["label"].to_numpy()
development_subjects = metadata.iloc[development_indices]["subject_id"].to_numpy()
reserved_subjects = set(metadata.iloc[partitions["test"]]["subject_id"])
if set(development_subjects) & reserved_subjects:
    raise ValueError("Reserved subjects entered Development.")
print("Six Development participant IDs:", sorted(set(development_subjects)))

# ================ 04. Run Three Participant-Disjoint CV Folds ================
fold_results, subject_results, summary = evaluate_group_cross_validation(
    features=development_features,
    labels=development_labels,
    group_ids=development_subjects,
    classes=TASK_CLASSES,
    n_splits=3,
    random_state=42,
)
display(summary.round(4))

display(fold_results[[
    "fold", "model", "train_subjects", "validation_subjects",
    "validation_classes_present", "macro_f1", "balanced_accuracy",
]].round(4))

**11. Compare Six-Person Fold and Subject-Level Scores**

Retain the fold-wise Macro F1 chart and per-participant fixed eight-class Macro F1 bar chart. Show both the fold mean and individual participant variation instead of overstating precision from two-person Validation folds.

In [ ]:
# ================= 05. Visualise Fold-Level Score Variation ==================
fig, ax = plt.subplots(figsize=(10, 5))

for model_name, table in fold_results.groupby("model", sort=False):
    table = table.sort_values("fold")
    ax.plot(
        table["fold"], table["macro_f1"], marker="o",
        label=f"{model_name} (mean={table['macro_f1'].mean():.3f})",
    )

ax.set(
    title="HARTH: subject-held-out development-fold Macro F1",
    xlabel="Held-out fold", ylabel="Macro F1",
    xticks=[1, 2, 3], ylim=(0, 1),
)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "02_six_dev_cross_validation_fold_macro_f1.png")
plt.show()


# ================== 06. Inspect Participant-Level Variation ==================

# ----------------------- 1. Participant-Level Results ------------------------
display(subject_results.sort_values(["model", "subject_id"]).round(4))

# ---------------------- 2. Visualise Fixed-Task Scores -----------------------
plot_data = subject_results.pivot(
    index="subject_id", columns="model", values="macro_f1_fixed_task"
).sort_index()
ax = plot_data.plot(kind="bar", figsize=(11, 5), rot=0)
ax.set(
    title="HARTH: fixed-task Macro F1 by held-out participant",
    xlabel="Validation participant", ylabel="Macro F1", ylim=(0, 1),
)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
save_figure(ax.figure, output_dir / "02_six_dev_cross_validation_participant_macro_f1.png")
plt.show()


# ------------------ 3. Save Six-Person Development Evidence ------------------
summary.to_csv(output_dir / "02_six_dev_model_summary.csv", index=False)
fold_results.to_csv(output_dir / "02_six_dev_fold_metrics.csv", index=False)
subject_results.to_csv(
    output_dir / "02_six_dev_participant_metrics.csv", index=False
)

**12. Reuse and Save Ten-Person Development Features**

Extend the original six-person Development set with S029, S030, S031 and S033, whose source features are already available in memory. Save one aligned feature-and-window-metadata CSV for subsequent full-Development work without repeating this computation.

In [ ]:
# ================ 12. Expand Development without Rewindowing =================

ten_source_mask = metadata_full["subject_id"].isin(DEVELOPMENT_SUBJECTS).to_numpy()
ten_meta_source = metadata_full.loc[ten_source_mask].reset_index(drop=True)
ten_features_source = features_full.loc[ten_source_mask].reset_index(drop=True)
ten_eligible = ten_meta_source["label"].isin(TASK_CLASSES).to_numpy()
metadata = ten_meta_source.loc[ten_eligible].reset_index(drop=True)
features = ten_features_source.loc[ten_eligible].reset_index(drop=True)
if set(metadata["subject_id"]) & RESERVED_TEST:
    raise ValueError("Reserved subjects entered ten-person Development.")
excluded = (ten_meta_source.loc[~ten_eligible]
    .groupby(["subject_id", "label"]).size()
    .rename("excluded_windows").reset_index()
    .rename(columns={"label": "source_code"}))
excluded["source_activity"] = excluded["source_code"].map(
    lambda code: ACTIVITY_NAMES.get(int(code), "UNDOCUMENTED source code")
)
overview = source_summary[source_summary["subject_id"].isin(
    DEVELOPMENT_SUBJECTS)].copy()
expanded_export = pd.concat([
    metadata.add_prefix("window_"), features
], axis=1)
expanded_export.to_csv(
    output_dir / "02_development10_statistical_features.csv", index=False
)
print("Ten-person Development features:", expanded_export.shape)
display(overview)

**13. Audit Excluded Codes and Ten-Person Activity Support**

Inspect excluded cycling and undocumented codes separately and plot the retained eight-class activity coverage per Development participant. These plots describe source representation, not model accuracy.

In [ ]:
# =============== 03. Review Source Label and Activity Support ================

# ---------------------- 1. Audit Excluded Source Labels ----------------------
print("Excluded windows by original source code:")
display(excluded.sort_values(["source_code", "subject_id"]))

# ----------------------- 2. Participant-Class Coverage -----------------------
class_coverage = pd.crosstab(
    metadata["subject_id"], metadata["label"]
).reindex(index=DEVELOPMENT_SUBJECTS, columns=TASK_CLASSES, fill_value=0)
class_coverage.columns = [
    f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES
]
display(class_coverage)

print("Classes present per subject:", (class_coverage > 0).sum(axis=1).to_dict())
print("Subjects per class:", (class_coverage > 0).sum(axis=0).to_dict())


# ================ 04. Visualise Participant Activity Coverage ================
fig, ax = plt.subplots(figsize=(13, 5.5))

heat = ax.imshow(class_coverage.to_numpy(), aspect="auto", cmap="YlGnBu")
ax.set_xticks(np.arange(len(class_coverage.columns)), class_coverage.columns, rotation=55, ha="right")
ax.set_yticks(np.arange(len(class_coverage.index)), class_coverage.index)
ax.set(title="HARTH expanded development cohort: windows per activity and subject",
       xlabel="Documented activity", ylabel="Development subject")
fig.colorbar(heat, ax=ax, label="Valid windows")
fig.tight_layout()
save_figure(fig, output_dir / "02_ten_dev_development_activity_coverage.png")
plt.show()

**14. Run Five Participant-Disjoint Ten-Person CV Folds**

Apply five GroupKFold splits with two held-out Development participants per fold. Refitting and Train-only scaling occur inside each fold. Reserved S023 and S032 are not included in the feature matrix.

In [ ]:
# ============== 05. Evaluate Five Participant-Disjoint CV Folds ==============
fold_results, subject_results, summary = evaluate_group_cross_validation(
    features=features.to_numpy(),
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    n_splits=5,
    random_state=42,
)
display(summary.round(4))
display(fold_results[[
    "fold", "model", "train_subjects", "validation_subjects",
    "validation_classes_present", "macro_f1", "balanced_accuracy"
]].round(4))

**15. Compare Expanded Fold and Participant Model Stability**

Visualise fold means and fixed-task Macro F1 per held-out participant. The six- and ten-person protocols use different participant sets and fold memberships, so their mean values are descriptive rather than paired comparisons.

In [ ]:
# ================ 06. Compare Fold and Subject-Level Results =================

# ----------------------- 1. Compare Fold-Level Scores ------------------------
fig, axes = plt.subplots(2, 1, figsize=(11, 9))

for name, part in fold_results.groupby("model", sort=False):
    part = part.sort_values("fold")
    axes[0].plot(part["fold"], part["macro_f1"], marker="o", label=name)

axes[0].set(title="Expanded development: held-out-fold Macro F1",
            xlabel="CV fold", ylabel="Macro F1", ylim=(0, 1))
axes[0].grid(alpha=0.25)
axes[0].legend()

# -------------------- 2. Compare Participant-Level Scores --------------------
per_subject = subject_results.pivot(
    index="subject_id", columns="model", values="macro_f1_fixed_task"
).sort_index()

per_subject.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set(title="Held-out-participant Macro F1 (eight-class denominator)",
            xlabel="Participant", ylabel="Macro F1", ylim=(0, 1))
axes[1].grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "02_ten_dev_development_cv_performance.png")
plt.show()
display(subject_results.sort_values(["model", "subject_id"]).round(4))

**16. Save Ten-Person Evidence and Package Current Outputs**

Save Stage 02 cohort, class-coverage and validation tables and package only the current Stage 02 output filenames. The expanded feature CSV is already written in Section 12 and will be consumed by the next Development stage.

In [ ]:
# =============== 16. Save Current Exploratory Research Outputs ===============

overview.to_csv(output_dir / "02_ten_dev_cohort_summary.csv", index=False)
excluded.to_csv(output_dir / "02_ten_dev_excluded_source_codes.csv", index=False)
class_coverage.rename_axis("subject_id").reset_index().to_csv(
    output_dir / "02_ten_dev_subject_activity_coverage.csv", index=False
)
summary.to_csv(output_dir / "02_ten_dev_model_summary.csv", index=False)
fold_results.to_csv(output_dir / "02_ten_dev_fold_metrics.csv", index=False)
subject_results.to_csv(
    output_dir / "02_ten_dev_participant_metrics.csv", index=False
)
print("Stage 02 research files:", output_dir)
print("Reusable ten-person feature CSV:",
      output_dir / "02_development10_statistical_features.csv")
download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="02"
)

**17. Scientific Interpretation and Next Research Boundary**

The early eight-person source audit, three-model pilot Validation, six-person Development cross-validation and expanded ten-person Development cross-validation are kept as distinct analyses. Source windows are built before restricting the supervised eight-class task. Unmapped codes are not silently relabelled or used in training.

**Interpretation:** Pilot Validation involves only two participants and the grouped CV procedures hold out two people per fold. Results should be interpreted alongside class coverage and per-person variation. Fixed-task Macro F1 includes all eight predefined activities; balanced accuracy may use only observed reference classes. Six- versus ten-person fold means are not a paired experiment because participant membership and fold allocation differ.

**Evaluation boundaries:** S023 and S032 remain outside Development model fitting and scoring. Earlier descriptive access to those subjects means no claim of strictly blind external Test evaluation follows from this stage.

**Next research stage:** Read the current 02_development10_statistical_features.csv from the Stage 02 output directory and add the remaining 19 Development participants in the next full-Development analysis. The saved ten-person windows should not be rebuilt without a scientific reason.